<a href="https://colab.research.google.com/github/A2-ashish/A2-ashish/blob/main/Kaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LinearRegression

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error

# ==============================
# 1. Load Kaggle Data
# ==============================

train_df = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")

# ==============================
# 2. Separate Target
# ==============================

X = train_df.drop(["SalePrice"], axis=1)
y = train_df["SalePrice"]

# ==============================
# 3. Automatic Column Detection
# ==============================

numeric_features = X.select_dtypes(include=np.number).columns
categorical_features = X.select_dtypes(include="object").columns

# ==============================
# 4. Numeric Pipeline
# ==============================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# ==============================
# 5. Categorical Pipeline
# ==============================

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# ==============================
# 6. Column Transformer
# ==============================

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# ==============================
# 7. Full Model Pipeline
# ==============================

model_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LinearRegression())
])

# ==============================
# 8. Train / Validation Split
# ==============================

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ==============================
# 9. Train Model
# ==============================

model_pipeline.fit(X_train, y_train)

# ==============================
# 10. Validation Score
# ==============================

val_pred = model_pipeline.predict(X_val)

rmsle = np.sqrt(mean_squared_log_error(y_val, val_pred))

print("Validation RMSLE Score:", rmsle)

# ==============================
# 11. Train Final Model on Full Data
# ==============================

model_pipeline.fit(X, y)

# ==============================
# 12. Predict Test Data
# ==============================

test_predictions = model_pipeline.predict(test_df)

# ==============================
# 13. Create Submission
# ==============================

submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": test_predictions
})

# ==============================
# 14. Save CSV
# ==============================

submission.to_csv("submission.csv", index=False)

print("submission.csv created!")